# QUELL — Step 01: Veri indirme

Uc dataset YPB'ye indirilir. Hepsi CSV feature surumu (ham pcap DEGIL).
- **CICIoT2023** — Kaggle `akashdogra/ciciot23csv` (birlesik 13 GB CSV)  ✅ onaylandi
- **Edge-IIoTset** — Kaggle `mohamedamineferrag/...` ; sadece ML/DNN CSV (hedefli)  ✅ onaylandi
- **N-BaIoT** — UCI id 442 (zip)  ⏳ calistirip teyit et

Onkosul: `~/.kaggle/kaggle.json` (kaggle.com > Settings > API > Legacy > kaggle.json indir, home'a yukle).

In [ ]:
# 1) Yollar + Kaggle
import os, subprocess, sys, glob
from pathlib import Path
ROOT = Path.home() / "quell-edge-llm-ids"
RAW  = ROOT / "data" / "raw"
for d in ["nbaiot","edge_iiotset","ciciot2023"]: (RAW/d).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable,"-m","pip","install","-q","kaggle"])
kj = Path.home()/"kaggle.json"; kd = Path.home()/".kaggle"; kd.mkdir(exist_ok=True)
if kj.exists():
    import shutil; shutil.copy(kj, kd/"kaggle.json"); os.chmod(kd/"kaggle.json",0o600); print("kaggle.json OK")
elif (kd/"kaggle.json").exists(): print("kaggle.json already in place OK")
else: print("WARNING: no kaggle.json - upload it to home, run the cell again.")
print("RAW:", RAW)

In [ ]:
# 2) CICIoT2023 (merged CSV, ~13 GB)  [APPROVED]
dst = RAW/"ciciot2023"
r = subprocess.run(f'kaggle datasets download -d akashdogra/ciciot23csv -p "{dst}" --unzip',
                   shell=True, capture_output=True, text=True)
print(r.stdout[-800:]); print("ERR:", r.stderr[-400:])
print(subprocess.run(f'ls -lh "{dst}"/*.csv', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 3) Edge-IIoTset — only ML + DNN CSV (targeted, raw traffic not fetched)  [APPROVED path]
slug = "mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot"
dst = RAW/"edge_iiotset"
targets = [
 "Edge-IIoTset dataset/Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv",
 "Edge-IIoTset dataset/Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv",
]
for t in targets:
    subprocess.run(f'kaggle datasets download -d {slug} -f "{t}" -p "{dst}"',
                   shell=True, capture_output=True, text=True)
subprocess.run(f'cd "{dst}" && for z in *.zip; do unzip -o -q "$z" && rm -f "$z"; done 2>/dev/null', shell=True)
print(subprocess.run(f'find "{dst}" -name "*.csv" -exec ls -lh {{}} +', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 4) N-BaIoT — UCI zip  [RUN & VERIFY]
dst = RAW/"nbaiot"
uci = "https://archive.ics.uci.edu/static/public/442/detection+of+iot+botnet+attacks+n+baiot.zip"
subprocess.run(f'wget -q -O "{dst}/nbaiot.zip" "{uci}"', shell=True)
z = dst/"nbaiot.zip"; print("zip size:", z.stat().st_size if z.exists() else "YOK")
if z.exists() and z.stat().st_size>10000:
    subprocess.run(f'cd "{dst}" && unzip -o -q nbaiot.zip', shell=True); print("extracted")
else:
    print("UCI failed -> Hugging Face fallback")
    subprocess.run(f'cd "{dst}" && git clone -q https://huggingface.co/datasets/codymlewis/nbaiot hf_nbaiot', shell=True)
print("csv sayisi:", subprocess.run(f'find "{dst}" -name "*.csv" | wc -l', shell=True, capture_output=True, text=True).stdout.strip())
print(subprocess.run(f'find "{dst}" -name "*.csv" | head', shell=True, capture_output=True, text=True).stdout)